# Stockiest April Slab Summary

Run this notebook from top to bottom to rebuild the final Excel output from `Sch utili.xlsx`.

Important rules implemented here:

- Use only `Distributor_Type = STOCKIEST DMS`.
- Use only April data.
- Exclude rows where `Scheme Name` contains `SAMT`.
- Split output into `12 ml` for `MRP = 15` and `18 ml` for `MRP = 20`.
- De-duplicate raw scheme rows into invoice lines before quantity and sale value calculations.
- Classify slabs at outlet level using each outlet's total non-duplicated quantity.
- Calculate `Total Discount %` as a quantity-weighted average of primary + secondary scheme percentage.


In [ ]:
from collections import Counter, defaultdict
from pathlib import Path

import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter


## 1. File paths

The notebook expects `Sch utili.xlsx` one folder above this repo folder.


In [ ]:
# If the notebook is opened from the repo folder, PROJECT_ROOT is the parent folder.
# If it is opened from the project root, PROJECT_ROOT remains the current folder.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "stockiest_april_slab_summary_repo" else NOTEBOOK_DIR

SOURCE = PROJECT_ROOT / "Sch utili.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "stockiest_april_slab_summary"
OUTPUT = OUTPUT_DIR / "stockiest_april_slab_summary_extended_samt_filtered.xlsx"

print(f"Source: {SOURCE}")
print(f"Output: {OUTPUT}")


## 2. Helper functions

These functions standardize values, define slabs, and format the final workbook.


In [ ]:
def norm(value):
    """Return a safe stripped string for comparisons."""
    return "" if value is None else str(value).strip()


def number(value):
    """Convert blank numeric cells to 0 and all other values to float."""
    if value is None or value == "":
        return 0.0
    return float(value)


def month_is_april(value):
    """Accept common April representations from the Month column."""
    text = norm(value).lower()
    return text in {"april", "apr", "4", "04"} or text.startswith("apr")


def slab_12ml(total_outlet_qty):
    """12 ml slab rules based on total outlet quantity."""
    if 8 <= total_outlet_qty < 144:
        return "Slab 1"
    if total_outlet_qty >= 144:
        return "Slab 2"
    return None


def slab_18ml(total_outlet_qty):
    """18 ml slab rules based on total outlet quantity."""
    if 8 <= total_outlet_qty < 32:
        return "Slab 1"
    if 32 <= total_outlet_qty < 576:
        return "Slab 2"
    if 576 <= total_outlet_qty < 960:
        return "Slab 3"
    if total_outlet_qty >= 960:
        return "Slab 4"
    return None


def empty_summary():
    """Accumulator used for each slab."""
    return {
        "qty": 0.0,
        "weighted_discount_sum": 0.0,
        "outlets": set(),
        "scheme_discount": 0.0,
        "distributor_sale_value": 0.0,
        "sale_value": 0.0,
    }


In [ ]:
def write_summary_sheet(wb, sheet_name, slab_names, summary):
    """Write one output sheet with six metrics for every slab."""
    ws = wb.create_sheet(sheet_name)

    headers = ["Month"]
    for slab in slab_names:
        headers.extend([
            f"{slab} Total Discount %",
            f"{slab} Total Quantity",
            f"{slab} Unique Store Count",
            f"{slab} Scheme Discount",
            f"{slab} Distributor Sale Value",
            f"{slab} Sale Value",
        ])
    ws.append(headers)

    row = ["April"]
    for slab in slab_names:
        data = summary[slab]
        weighted_discount = data["weighted_discount_sum"] / data["qty"] if data["qty"] else 0
        row.extend([
            weighted_discount,
            data["qty"],
            len(data["outlets"]),
            data["scheme_discount"],
            data["distributor_sale_value"],
            data["sale_value"],
        ])
    ws.append(row)

    # Simple readable formatting only.
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for cell in ws[2]:
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for col_idx in range(1, ws.max_column + 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = 22
    ws.column_dimensions["A"].width = 12

    for col_idx in range(2, ws.max_column + 1, 6):
        ws.cell(2, col_idx).number_format = "0.00"
        ws.cell(2, col_idx + 1).number_format = "0"
        ws.cell(2, col_idx + 2).number_format = "0"
        ws.cell(2, col_idx + 3).number_format = "#,##0.00"
        ws.cell(2, col_idx + 4).number_format = "#,##0.00"
        ws.cell(2, col_idx + 5).number_format = "#,##0.00"


## 3. Load source data and validate columns

The workbook must contain all columns used in the calculation. The cell below stops early if any required column is missing.


In [ ]:
source_wb = openpyxl.load_workbook(SOURCE, read_only=True, data_only=True)
source_ws = source_wb.active
headers = [cell.value for cell in next(source_ws.iter_rows(min_row=1, max_row=1))]
idx = {name: i for i, name in enumerate(headers)}

required_columns = [
    "Month", "Distributor_Type", "Distributor code", "Distributor_id", "Outlet_Id",
    "Bill No", "Invoice Date", "Invoice qty. pieces", "skunitid", "skucode",
    "Batch_Id", "Batch No", "MRP", "Scheme Name", "Scheme_Reason", "%_Scheme",
    "Scheme_discount", "Distributor_Sale_Value", "Sale_Value",
]
missing_columns = [name for name in required_columns if name not in idx]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"Loaded sheet: {source_ws.title}")
print(f"Rows including header: {source_ws.max_row:,}")
print(f"Columns: {len(headers)}")


## 4. De-duplicate raw rows into invoice lines

Raw data can have separate rows for primary and secondary scheme on the same invoice line.

This cell first filters the raw rows, then combines them into invoice-line records. Quantity and sale values are kept once per invoice line, while `Scheme_discount` is summed across the scheme rows.


In [ ]:
line_key_columns = [
    "Distributor code", "Distributor_id", "Outlet_Id", "Bill No", "Invoice Date",
    "skunitid", "skucode", "Batch_Id", "Batch No", "MRP",
]
line_key_indexes = [idx[name] for name in line_key_columns]

lines = {}
counts = Counter()
reason_by_mrp = Counter()

for row in source_ws.iter_rows(min_row=2, values_only=True):
    # Base filter: Stockiest only.
    if norm(row[idx["Distributor_Type"]]) != "STOCKIEST DMS":
        continue
    counts["stockiest_rows"] += 1

    # Month filter: April only.
    if not month_is_april(row[idx["Month"]]):
        continue
    counts["april_stockiest_rows"] += 1

    # Requested exclusion: remove any scheme whose name contains SAMT.
    if "SAMT" in norm(row[idx["Scheme Name"]]).upper():
        counts["samt_rows_excluded"] += 1
        continue

    # Pack filter: MRP 15 = 12 ml, MRP 20 = 18 ml.
    mrp = number(row[idx["MRP"]])
    if mrp not in {15.0, 20.0}:
        counts["other_mrp_rows_excluded"] += 1
        continue

    reason = norm(row[idx["Scheme_Reason"]])
    reason_by_mrp[(mrp, reason)] += 1

    # This key identifies one invoice line. Repeated scheme rows collapse into this one record.
    line_key = tuple(row[i] for i in line_key_indexes)
    if line_key not in lines:
        lines[line_key] = {
            "mrp": mrp,
            "qty": number(row[idx["Invoice qty. pieces"]]),
            "outlet": row[idx["Outlet_Id"]],
            "primary": 0.0,
            "secondary": 0.0,
            "scheme_discount": 0.0,
            "distributor_sale_value": number(row[idx["Distributor_Sale_Value"]]),
            "sale_value": number(row[idx["Sale_Value"]]),
        }

    scheme_pct = number(row[idx["%_Scheme"]])

    # Primary scheme mapping differs by pack.
    if mrp == 15.0 and reason == "Non-Claimable_Discount":
        lines[line_key]["primary"] += scheme_pct
    elif mrp == 20.0 and reason == "Non-Claimable_Free-Quantity":
        lines[line_key]["primary"] += scheme_pct
    elif reason == "Claimable":
        lines[line_key]["secondary"] += scheme_pct

    # Scheme_discount is a rupee/value amount by scheme row, so it is additive.
    lines[line_key]["scheme_discount"] += number(row[idx["Scheme_discount"]])

print(dict(counts))
print(f"Invoice lines after filters: {len(lines):,}")
print({f"MRP {mrp:g} | {reason}": count for (mrp, reason), count in reason_by_mrp.items()})


## 5. Assign slabs at outlet level

Slabs are not assigned per invoice. They are assigned after summing total quantity for each `Outlet_Id` within each pack/MRP.

After an outlet is assigned to a slab, all invoice lines for that outlet and pack are rolled into that slab.


In [ ]:
# Total non-duplicated quantity per outlet within each pack.
outlet_qty = defaultdict(float)
for line in lines.values():
    outlet_qty[(line["mrp"], line["outlet"])] += line["qty"]

summaries = {
    "12 ml": defaultdict(empty_summary),
    "18 ml": defaultdict(empty_summary),
}
below_minimum_outlets = set()

for line in lines.values():
    total_outlet_qty = outlet_qty[(line["mrp"], line["outlet"])]

    if line["mrp"] == 15.0:
        sheet_name = "12 ml"
        slab = slab_12ml(total_outlet_qty)
    else:
        sheet_name = "18 ml"
        slab = slab_18ml(total_outlet_qty)

    # Outlets with total quantity below 8 are outside all requested slabs.
    if slab is None:
        below_minimum_outlets.add((line["mrp"], line["outlet"]))
        continue

    total_discount_pct = line["primary"] + line["secondary"]
    target = summaries[sheet_name][slab]

    target["qty"] += line["qty"]
    target["weighted_discount_sum"] += total_discount_pct * line["qty"]
    target["outlets"].add(line["outlet"])
    target["scheme_discount"] += line["scheme_discount"]
    target["distributor_sale_value"] += line["distributor_sale_value"]
    target["sale_value"] += line["sale_value"]

print(f"Outlet/MRP groups after filters: {len(outlet_qty):,}")
print(f"Outlet/MRP groups below slab minimum: {len(below_minimum_outlets):,}")


## 6. Create final Excel workbook

Each sheet has one April row. Each slab has six metrics: weighted discount %, quantity, unique store count, scheme discount, distributor sale value, and sale value.


In [ ]:
output_wb = Workbook()
output_wb.remove(output_wb.active)

write_summary_sheet(output_wb, "12 ml", ["Slab 1", "Slab 2"], summaries["12 ml"])
write_summary_sheet(output_wb, "18 ml", ["Slab 1", "Slab 2", "Slab 3", "Slab 4"], summaries["18 ml"])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_wb.save(OUTPUT)

print(f"Saved: {OUTPUT}")


## 7. Validate saved workbook

This validation rereads the saved Excel file and compares every metric against the in-memory calculation. If anything differs beyond rounding tolerance, this cell raises an error.


In [ ]:
check_wb = openpyxl.load_workbook(OUTPUT, data_only=True, read_only=True)
differences = []
final_rows = {}

for sheet_name, slabs in {"12 ml": ["Slab 1", "Slab 2"], "18 ml": ["Slab 1", "Slab 2", "Slab 3", "Slab 4"]}.items():
    check_ws = check_wb[sheet_name]
    output_headers = [cell.value for cell in next(check_ws.iter_rows(min_row=1, max_row=1))]
    output_values = [cell.value for cell in next(check_ws.iter_rows(min_row=2, max_row=2))]
    output_row = dict(zip(output_headers, output_values))
    final_rows[sheet_name] = output_row

    for slab in slabs:
        data = summaries[sheet_name][slab]
        expected_values = {
            f"{slab} Total Discount %": data["weighted_discount_sum"] / data["qty"] if data["qty"] else 0,
            f"{slab} Total Quantity": data["qty"],
            f"{slab} Unique Store Count": len(data["outlets"]),
            f"{slab} Scheme Discount": data["scheme_discount"],
            f"{slab} Distributor Sale Value": data["distributor_sale_value"],
            f"{slab} Sale Value": data["sale_value"],
        }
        for header, expected in expected_values.items():
            actual = number(output_row.get(header))
            if abs(actual - expected) > 0.01:
                differences.append((sheet_name, header, expected, actual))

if differences:
    raise AssertionError(f"Saved workbook does not match calculation: {differences[:5]}")

print("Validation passed: saved workbook matches calculated values.")
final_rows


## 8. Pack-level sales split

This additional output treats sales as non-duplicated `Invoice qty. pieces` and summarizes it at pack level only, without slabs.

It calculates:

- Stockiest vs Super Stockiest quantity split for `12 ml` and `18 ml`.
- Within Stockiest, SAMT vs Non-SAMT quantity split for each pack.

SAMT classification uses `Scheme Name` containing `SAMT`. The calculation checks for mixed invoice lines where the same de-duplicated line has both SAMT and non-SAMT rows.


In [ ]:
SALES_SPLIT_OUTPUT = OUTPUT_DIR / "pack_level_sales_split.xlsx"

# Re-open the source workbook for a clean pass over raw rows.
sales_wb = openpyxl.load_workbook(SOURCE, read_only=True, data_only=True)
sales_ws = sales_wb.active
sales_headers = [cell.value for cell in next(sales_ws.iter_rows(min_row=1, max_row=1))]
sales_idx = {name: i for i, name in enumerate(sales_headers)}

sales_line_key_columns = [
    "Distributor_Type", "Distributor code", "Distributor_id", "Outlet_Id", "Bill No",
    "Invoice Date", "skunitid", "skucode", "Batch_Id", "Batch No", "MRP",
]
sales_line_key_indexes = [sales_idx[name] for name in sales_line_key_columns]

sales_lines = {}
sales_counts = Counter()

for row in sales_ws.iter_rows(min_row=2, values_only=True):
    if not month_is_april(row[sales_idx["Month"]]):
        continue

    distributor_type = norm(row[sales_idx["Distributor_Type"]])
    if distributor_type not in {"STOCKIEST DMS", "SUPER STOCKIEST DMS"}:
        continue

    mrp = number(row[sales_idx["MRP"]])
    if mrp not in {15.0, 20.0}:
        sales_counts["other_mrp_rows_excluded"] += 1
        continue

    line_key = tuple(row[i] for i in sales_line_key_indexes)
    if line_key not in sales_lines:
        sales_lines[line_key] = {
            "distributor_type": distributor_type,
            "pack": "12 ml" if mrp == 15.0 else "18 ml",
            "qty": number(row[sales_idx["Invoice qty. pieces"]]),
            "has_samt": False,
            "has_non_samt": False,
        }

    is_samt = "SAMT" in norm(row[sales_idx["Scheme Name"]]).upper()
    sales_lines[line_key]["has_samt"] = sales_lines[line_key]["has_samt"] or is_samt
    sales_lines[line_key]["has_non_samt"] = sales_lines[line_key]["has_non_samt"] or not is_samt

pack_distributor_qty = defaultdict(lambda: defaultdict(float))
stockiest_samt_qty = defaultdict(lambda: defaultdict(float))
stockiest_line_type_counts = Counter()

for line in sales_lines.values():
    pack = line["pack"]
    distributor_type = line["distributor_type"]
    qty = line["qty"]

    pack_distributor_qty[pack][distributor_type] += qty

    if distributor_type == "STOCKIEST DMS":
        if line["has_samt"] and line["has_non_samt"]:
            bucket = "Mixed SAMT/non-SAMT"
        elif line["has_samt"]:
            bucket = "SAMT"
        else:
            bucket = "Non-SAMT"
        stockiest_samt_qty[pack][bucket] += qty
        stockiest_line_type_counts[(pack, bucket)] += 1


In [ ]:
sales_split_rows = []
for pack in ["12 ml", "18 ml"]:
    stockiest_qty = pack_distributor_qty[pack]["STOCKIEST DMS"]
    super_stockiest_qty = pack_distributor_qty[pack]["SUPER STOCKIEST DMS"]
    total_qty = stockiest_qty + super_stockiest_qty
    samt_qty = stockiest_samt_qty[pack]["SAMT"]
    non_samt_qty = stockiest_samt_qty[pack]["Non-SAMT"]
    mixed_qty = stockiest_samt_qty[pack]["Mixed SAMT/non-SAMT"]

    sales_split_rows.append({
        "Pack": pack,
        "Stockiest Qty": stockiest_qty,
        "Super Stockiest Qty": super_stockiest_qty,
        "Total Qty": total_qty,
        "Stockiest Share %": stockiest_qty / total_qty * 100 if total_qty else 0,
        "Super Stockiest Share %": super_stockiest_qty / total_qty * 100 if total_qty else 0,
        "Stockiest SAMT Qty": samt_qty,
        "Stockiest Non-SAMT Qty": non_samt_qty,
        "Stockiest Mixed Qty": mixed_qty,
        "Stockiest SAMT Share %": samt_qty / stockiest_qty * 100 if stockiest_qty else 0,
        "Stockiest Non-SAMT Share %": non_samt_qty / stockiest_qty * 100 if stockiest_qty else 0,
    })

sales_split_rows


In [ ]:
sales_output_wb = Workbook()
sales_output_ws = sales_output_wb.active
sales_output_ws.title = "Pack Sales Split"

sales_headers = [
    "Pack", "Stockiest Qty", "Super Stockiest Qty", "Total Qty",
    "Stockiest Share %", "Super Stockiest Share %",
    "Stockiest SAMT Qty", "Stockiest Non-SAMT Qty",
    "Stockiest SAMT Share %", "Stockiest Non-SAMT Share %",
]
sales_output_ws.append(sales_headers)

for row in sales_split_rows:
    sales_output_ws.append([
        row["Pack"], row["Stockiest Qty"], row["Super Stockiest Qty"], row["Total Qty"],
        row["Stockiest Share %"], row["Super Stockiest Share %"],
        row["Stockiest SAMT Qty"], row["Stockiest Non-SAMT Qty"],
        row["Stockiest SAMT Share %"], row["Stockiest Non-SAMT Share %"],
    ])

for cell in sales_output_ws[1]:
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center", wrap_text=True)
for row in sales_output_ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(horizontal="center")
for col_idx in range(1, sales_output_ws.max_column + 1):
    sales_output_ws.column_dimensions[get_column_letter(col_idx)].width = 22
for col_idx in [5, 6, 9, 10]:
    for row_idx in range(2, sales_output_ws.max_row + 1):
        sales_output_ws.cell(row_idx, col_idx).number_format = "0.00"
for col_idx in [2, 3, 4, 7, 8]:
    for row_idx in range(2, sales_output_ws.max_row + 1):
        sales_output_ws.cell(row_idx, col_idx).number_format = "#,##0"

sales_output_wb.save(SALES_SPLIT_OUTPUT)

print(f"Saved: {SALES_SPLIT_OUTPUT}")
print(f"Unique invoice lines used for sales split: {len(sales_lines):,}")
print({f"{pack} | {bucket}": count for (pack, bucket), count in stockiest_line_type_counts.items()})
